# 📘 Module 2.3 — Object Detection for ADAS

**Goal:** Understand how object detection works and apply it to ADAS scenarios.

## Object Detection in ADAS
Object detection answers: **What** objects are in the image and **where** are they?

```
Classification:  "This image contains a car"      → one label
Detection:       "Car at (x1,y1,x2,y2), conf=0.95" → boxes + labels
Segmentation:    "These pixels belong to a car"     → pixel-level
```

### Key Detection Architectures:
| Model | Type | Speed | Accuracy | ADAS Use |
|-------|------|-------|----------|----------|
| Faster R-CNN | Two-stage | Slow | High | Research |
| SSD | One-stage | Fast | Medium | Embedded |
| YOLO | One-stage | Very Fast | High | Production |
| DETR | Transformer | Medium | High | Research |

---

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

## 1. Bounding Boxes — The Core Representation

A bounding box defines a rectangular region: `[x_min, y_min, x_max, y_max]`

```
(x1, y1) ┌──────────────┐
          │              │
          │   OBJECT     │
          │              │
          └──────────────┘ (x2, y2)
```

In [ ]:
# --- Bounding Box Operations ---

def compute_iou(box1, box2):
    """Compute Intersection over Union (IoU) between two boxes.
    
    IoU is the KEY metric for object detection:
    - IoU > 0.5 → correct detection (standard threshold)
    - IoU > 0.7 → strict matching
    
    Args:
        box1, box2: [x1, y1, x2, y2] format
    """
    # Intersection coordinates
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Intersection area
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # Union area
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

# Example: predicted vs. ground truth box
gt_box = [100, 100, 300, 250]    # Ground truth
pred_box = [120, 110, 310, 260]  # Prediction
bad_pred = [400, 400, 500, 500]  # Bad prediction

print(f"Good prediction IoU: {compute_iou(gt_box, pred_box):.4f}")
print(f"Bad prediction IoU:  {compute_iou(gt_box, bad_pred):.4f}")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.set_xlim(0, 600)
ax.set_ylim(0, 400)
ax.invert_yaxis()

gt_rect = patches.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                            linewidth=2, edgecolor='green', facecolor='green', alpha=0.2)
pred_rect = patches.Rectangle((pred_box[0], pred_box[1]), pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                              linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.2)
ax.add_patch(gt_rect)
ax.add_patch(pred_rect)
ax.legend(['Ground Truth', 'Prediction'])
ax.set_title(f'IoU = {compute_iou(gt_box, pred_box):.3f}')
plt.show()

## 2. Non-Maximum Suppression (NMS)

Detection models often produce **many overlapping boxes** for the same object. NMS keeps only the best one.

```
Before NMS: 100 overlapping boxes per car
After NMS:  1 clean box per car
```

In [ ]:
def non_max_suppression(boxes, scores, iou_threshold=0.5):
    """Simple NMS implementation."""
    # Sort by confidence (highest first)
    order = scores.argsort()[::-1]
    keep = []
    
    while len(order) > 0:
        # Keep the highest confidence box
        best_idx = order[0]
        keep.append(best_idx)
        
        # Remove boxes with high IoU overlap
        remaining = []
        for idx in order[1:]:
            if compute_iou(boxes[best_idx], boxes[idx]) < iou_threshold:
                remaining.append(idx)
        
        order = np.array(remaining)
    
    return keep

# Example: multiple detections for one car
boxes = [
    [100, 100, 250, 200],  # High overlap
    [110, 105, 260, 210],  # High overlap
    [105, 98, 255, 205],   # High overlap
    [400, 150, 550, 300],  # Different object
    [410, 155, 560, 305],  # Overlap with above
]
scores = np.array([0.9, 0.85, 0.75, 0.95, 0.6])

kept = non_max_suppression(boxes, scores, iou_threshold=0.3)
print(f"Before NMS: {len(boxes)} boxes")
print(f"After NMS:  {len(kept)} boxes (indices: {kept})")
print(f"Kept boxes: {[boxes[i] for i in kept]}")
print(f"Kept scores: {[scores[i] for i in kept]}")

## 3. Anchor Boxes

Most detectors use **anchor boxes** — predefined box templates at each position.

```
At each position, try different:
- Scales: small, medium, large
- Aspect ratios: tall (pedestrian), wide (car), square

The model predicts OFFSETS from these anchors.
```

In [ ]:
def generate_anchors(feature_map_size, scales, aspect_ratios, image_size=416):
    """Generate anchor boxes for a feature map."""
    stride = image_size / feature_map_size
    anchors = []
    
    for i in range(feature_map_size):
        for j in range(feature_map_size):
            cx = (j + 0.5) * stride
            cy = (i + 0.5) * stride
            for scale in scales:
                for ratio in aspect_ratios:
                    w = scale * np.sqrt(ratio)
                    h = scale / np.sqrt(ratio)
                    anchors.append([cx - w/2, cy - h/2, cx + w/2, cy + h/2])
    
    return np.array(anchors)

# Generate anchors for a 13x13 feature map (like YOLO)
anchors = generate_anchors(
    feature_map_size=13,
    scales=[32, 64, 128],
    aspect_ratios=[0.5, 1.0, 2.0],
    image_size=416
)
print(f"Total anchors: {len(anchors)}")
print(f"Per cell: {3 * 3} = 3 scales × 3 ratios")

# Visualize anchors at center
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 416)
ax.set_ylim(416, 0)

center_anchors = anchors[len(anchors)//2 - 4: len(anchors)//2 + 5]
colors = plt.cm.Set1(np.linspace(0, 1, len(center_anchors)))

for anchor, color in zip(center_anchors, colors):
    rect = patches.Rectangle(
        (anchor[0], anchor[1]), anchor[2]-anchor[0], anchor[3]-anchor[1],
        linewidth=2, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)

ax.plot(208, 208, 'r+', markersize=15, markeredgewidth=3)
ax.set_title('Anchor Boxes at Center (Different Scales & Ratios)')
ax.grid(True, alpha=0.3)
plt.show()

## 4. Using a Pretrained Detection Model

Let's use Faster R-CNN pretrained on COCO (80 object classes including cars, people, trucks).

In [ ]:
# COCO class names (subset relevant to ADAS)
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle',
    'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'N/A', 'stop sign', 'parking meter', 'bench',
    # ... more classes
]

ADAS_RELEVANT = {'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck',
                 'traffic light', 'stop sign'}

print("ADAS-relevant COCO classes:")
for cls in ADAS_RELEVANT:
    print(f"  • {cls}")

# Load pretrained Faster R-CNN (uncomment to download ~160MB)
# model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
#     weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.COCO_V1
# )
# model.eval()

print("\n💡 Uncomment above to load pretrained Faster R-CNN")
print("The model can detect all ADAS-relevant objects out of the box!")

In [ ]:
# --- Simulated Detection Output ---
# This is what a detection model outputs:

# Simulated detections on a driving scene
detections = {
    'boxes': torch.tensor([
        [150, 200, 350, 400],  # car
        [400, 180, 500, 350],  # car
        [50, 250, 100, 400],   # pedestrian
        [550, 150, 620, 200],  # traffic light
    ], dtype=torch.float32),
    'labels': ['car', 'car', 'person', 'traffic light'],
    'scores': [0.95, 0.88, 0.92, 0.78],
}

# Visualize detections
fig, ax = plt.subplots(figsize=(10, 6))
# Create a fake scene (gradient for sky/road)
scene = np.zeros((480, 640, 3), dtype=np.uint8)
scene[:240, :] = [135, 206, 235]  # Sky
scene[240:, :] = [100, 100, 100]  # Road
ax.imshow(scene)

colors = {'car': 'lime', 'person': 'red', 'traffic light': 'yellow'}

for box, label, score in zip(detections['boxes'], detections['labels'], detections['scores']):
    x1, y1, x2, y2 = box.numpy()
    color = colors.get(label, 'white')
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                            linewidth=2, edgecolor=color, facecolor=color, alpha=0.3)
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"{label} {score:.0%}",
            color=color, fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.set_title('Object Detection Output (Simulated ADAS Scene)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Detection Metrics

| Metric | What it measures |
|--------|------------------|
| **Precision** | Of all detections, how many are correct? |
| **Recall** | Of all objects, how many did we find? |
| **mAP** (mean Average Precision) | Overall detection quality |
| **FPS** | Speed (critical for real-time ADAS) |

In [ ]:
# --- Precision-Recall Example ---

# Simulated scenario:
# Ground truth: 5 cars in the image
# Model detected: 6 boxes
# 4 correct (IoU > 0.5), 2 false positives

true_positives = 4
false_positives = 2
false_negatives = 1  # 1 car missed

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
f1 = 2 * precision * recall / (precision + recall)

print(f"True Positives:  {true_positives} (correct detections)")
print(f"False Positives: {false_positives} (phantom detections)")
print(f"False Negatives: {false_negatives} (missed objects)")
print(f"")
print(f"Precision: {precision:.2%} (of detections, how many are real?)")
print(f"Recall:    {recall:.2%} (of real objects, how many found?)")
print(f"F1 Score:  {f1:.2%}")
print(f"")
print("⚠️  In ADAS, high RECALL is critical — missing a pedestrian is dangerous!")

---
## ✅ Key Takeaways

1. **Object detection** = classification + localization (bounding boxes)
2. **IoU** measures how well predicted boxes match ground truth
3. **NMS** removes duplicate detections for the same object
4. **Anchor boxes** provide templates that the model refines
5. **YOLO** is preferred for real-time ADAS (fast + accurate)
6. **Recall** is more important than precision in safety-critical ADAS

---
## 📖 Next Steps
➡️ **Next module:** [03_RNNs/01_rnn_fundamentals.ipynb](../03_RNNs/01_rnn_fundamentals.ipynb) — Recurrent Neural Networks for temporal data